In [1]:
# Lecture 9
# import packages
from scipy.stats import norm
from scipy.stats import chi2
from scipy import stats

from statsmodels.formula.api import glm
from statsmodels.genmod.families import Poisson
from statsmodels.stats.contingency_tables import Table2x2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
# os.chdir(r'C:\Users\Linda\Desktop\ST5213py')
pd.set_option("display.width", 2000)

# import custom functions
from ST5213_helpers import *

# fig_save_dir = r'C:/Users/Linda/OneDrive - National University of Singapore/ST5213 2627/Lectures/Lecture_9/figure'

In [2]:
# read Jane Austen data
df = pd.read_csv('dataset/JaneAusten.txt', sep=' ')
print(df.head(), '\n')

# convert Book and Word into categorical variables
df['Book'] = pd.Categorical(df['Book'])
df['Word'] = pd.Categorical(df['Word'])
print(df.info())

   Count               Book  Word
0    147  Sense&Sensibility     a
1     25  Sense&Sensibility    an
2     32  Sense&Sensibility  this
3     94  Sense&Sensibility  that
4     59  Sense&Sensibility  with 

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   Count   24 non-null     int64   
 1   Book    24 non-null     category
 2   Word    24 non-null     category
dtypes: category(2), int64(1)
memory usage: 452.0 bytes
None


In [3]:
fm3 = glm(formula='Count ~ Word + Book', data=df, family=Poisson()).fit()
fm4 = update_glm(fm3, add='Word:Book')   
print(anova_nested(fm3, fm4))

fm5 = glm(formula='Count ~ Word + Book', data=df.loc[df['Book'].isin(['SanditionI', 'SanditionII'])], family=Poisson()).fit()
fm6 = update_glm(fm5, add='Word:Book')   
print(anova_nested(fm5, fm6))

         Resid. Df    Resid. Dev    Df   Deviance  Pr(>Chi)
Model 1       15.0  4.432405e+01   NaN        NaN       NaN
Model 2        0.0 -4.929390e-14  15.0  44.324052  0.000098
         Resid. Df    Resid. Dev   Df  Deviance  Pr(>Chi)
Model 1        5.0  1.977684e+01  NaN       NaN       NaN
Model 2        0.0  6.661338e-15  5.0  19.77684  0.001376


c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\regression\_tools.py:133: RuntimeWarning: divide by zero encountered in scalar divide
  scale = np.dot(wresid, wresid) / df_resid
c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1486: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = wls_model.fit(method=wls_method2)
c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\st

In [4]:
# Compare observed and fitted values of fm4
tab = df.pivot(index='Word', columns='Book', values='Count')
tab_fm4 = np.array(fm4.fittedvalues).reshape(4, 6).transpose()
print(tab )
print(tab_fm4)

Book     Emma  SanditionI  SanditionII  Sense&Sensibility
Word                                                     
a         186         101           83                147
an         26          11           29                 25
that      105          37           22                 94
this       39          15           15                 32
with       74          28           43                 59
without    10          10            4                 18
[[147. 186. 101.  83.]
 [ 25.  26.  11.  29.]
 [ 32.  39.  15.  15.]
 [ 94. 105.  37.  22.]
 [ 59.  74.  28.  43.]
 [ 18.  10.  10.   4.]]


In [5]:
# reorder the lvels in Book and Word for ease of comparison
df['Book'] = pd.Categorical(df['Book'], categories=['Sense&Sensibility', 'Emma', 'SanditionI', 'SanditionII'], ordered=True)
df['Word'] = pd.Categorical(df['Word'], categories=['a', 'an', 'this', 'that', 'with', 'without'], ordered=True)

In [6]:
# Compare observed and fitted values of fm4
tab = df.pivot(index='Word', columns='Book', values='Count')
tab_fm4 = np.array(fm4.fittedvalues).reshape(4, 6).transpose()
print(tab )
print(tab_fm4)

Book     Sense&Sensibility  Emma  SanditionI  SanditionII
Word                                                     
a                      147   186         101           83
an                      25    26          11           29
this                    32    39          15           15
that                    94   105          37           22
with                    59    74          28           43
without                 18    10          10            4
[[147. 186. 101.  83.]
 [ 25.  26.  11.  29.]
 [ 32.  39.  15.  15.]
 [ 94. 105.  37.  22.]
 [ 59.  74.  28.  43.]
 [ 18.  10.  10.   4.]]


In [7]:
def compute_marginals(tab):
    new_col = tab.sum(axis=1)
    tab = np.hstack([tab, new_col.reshape(-1,1)])
    new_row = tab.sum(axis=0)
    tab = np.vstack([tab, new_row])
    return tab 

# Compare observed and fitted values of fm3
tab_fm3 = np.array(fm3.fittedvalues).reshape(4, 6).transpose()
print(compute_marginals(np.array(tab)), '\n')
print(np.round(compute_marginals(tab_fm3), 2))

[[ 147  186  101   83  517]
 [  25   26   11   29   91]
 [  32   39   15   15  101]
 [  94  105   37   22  258]
 [  59   74   28   43  204]
 [  18   10   10    4   42]
 [ 375  440  202  196 1213]] 

[[ 159.83  187.54   86.1    83.54  517.  ]
 [  28.13   33.01   15.15   14.7    91.  ]
 [  31.22   36.64   16.82   16.32  101.  ]
 [  79.76   93.59   42.96   41.69  258.  ]
 [  63.07   74.     33.97   32.96  204.  ]
 [  12.98   15.23    6.99    6.79   42.  ]
 [ 375.    440.    202.    196.   1213.  ]]


In [8]:
# Compute statistics for fm3
O = df['Count']
E = fm3.fittedvalues

# Pearson chisq 
print( ((O - E)**2 / E).sum() )

# Likelihood-ratio 
print(2 * (O * np.log(O/E)).sum())
print(fm3.deviance)

45.57751162922614
44.324052399386105
44.324052401638454


In [9]:
# Wermuth data 
df = pd.read_csv('dataset/Wermuth.txt', sep=' ')
print(df, '\n')

# Assign multiple variables as categorical
df['I'] = pd.Categorical(df['I'], categories=['no', 'yes'], ordered=True)
df['A'] = pd.Categorical(df['A'], categories=['<30', '30+'], ordered=True)
df['G'] = pd.Categorical(df['G'], categories=['<=260', '>260'], ordered=True)
df['S'] = pd.Categorical(df['S'], categories=['<5', '5+'], ordered=True)
print(df.info()) 

      A   S      G    I  Count
0   <30  <5  <=260   no     50
1   <30  <5   >260   no     24
2   <30  5+  <=260   no      9
3   <30  5+   >260   no      6
4   30+  <5  <=260   no     41
5   30+  <5   >260   no     14
6   30+  5+  <=260   no      4
7   30+  5+   >260   no      1
8   <30  <5  <=260  yes    315
9   <30  <5   >260  yes   4012
10  <30  5+  <=260  yes     40
11  <30  5+   >260  yes    459
12  30+  <5  <=260  yes    147
13  30+  <5   >260  yes   1594
14  30+  5+  <=260  yes     11
15  30+  5+   >260  yes    124 

<class 'pandas.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   A       16 non-null     category
 1   S       16 non-null     category
 2   G       16 non-null     category
 3   I       16 non-null     category
 4   Count   16 non-null     int64   
dtypes: category(4), int64(1)
memory usage: 820.0 bytes
None


In [10]:
# Fit loglinear model and perform model selection
fm = glm(formula='Count ~ (A + S + G + I)**4', data=df, family=Poisson()).fit()
print(drop1_glm(fm))

fm = update_glm(fm, drop='A:S:G:I')
print(drop1_glm(fm))

fm = update_glm(fm, drop='A:S:G')
print(drop1_glm(fm))

fm = update_glm(fm, drop='S:G:I')
print(drop1_glm(fm))

fm = update_glm(fm, drop='S:G')
print(drop1_glm(fm))

fm = update_glm(fm, drop='A:G:I')
print(drop1_glm(fm))

fm = update_glm(fm, drop='A:S:I')
print(drop1_glm(fm))

fm = update_glm(fm, drop='A:G')
print(drop1_glm(fm))

fm = update_glm(fm, drop='S:I')
print(drop1_glm(fm))

        Df      Deviance         AIC      LRT  Pr(>Chi)
<none>      6.048495e-13  123.973219                   
A:S:G:I  1  3.593495e-01  122.332569  0.35935  0.548868
       Df  Deviance         AIC       LRT  Pr(>Chi)
<none>     0.359350  122.332569                    
A:S:G   1  0.411333  120.384552  0.051983  0.819648
A:S:I   1  0.694036  120.667255  0.334686  0.562913
A:G:I   1  0.809179  120.782398   0.44983  0.502416
S:G:I   1  0.585752  120.558971  0.226403  0.634204
       Df  Deviance         AIC       LRT  Pr(>Chi)
<none>     0.411333  120.384552                    
A:S:I   1  0.918132  118.891351  0.506799  0.476528
A:G:I   1  0.853175  118.826394  0.441842  0.506235
S:G:I   1  0.638382  118.611601  0.227049  0.633721
       Df  Deviance         AIC       LRT  Pr(>Chi)
<none>     0.638382  118.611601                    
S:G     1  0.750760  116.723979  0.112378  0.737453
A:S:I   1  1.205726  117.178945  0.567344  0.451316
A:G:I   1  1.162712  117.135931   0.52433     0.469


c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\regression\_tools.py:133: RuntimeWarning: divide by zero encountered in scalar divide
  scale = np.dot(wresid, wresid) / df_resid
c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(


In [11]:
# Goodness of fit test 
print(fm.summary2())
print(chi2.sf(fm.deviance, fm.df_resid))

                 Results: Generalized linear model
Model:                GLM                Method:           IRLS    
Model Family:         Poisson            AIC:              115.6929
Link Function:        Log                BIC:              121.8737
Dependent Variable:   Count              Log-Likelihood:   -49.846 
Date:                 2026-10-13 10:01   LL-Null:          -10202. 
No. Observations:     16                 Deviance:         7.7197  
Df Model:             7                  Pearson chi2:     8.73    
Df Residuals:         8                  Scale:            1.0000  
-------------------------------------------------------------------
                    Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
-------------------------------------------------------------------
Intercept           4.0186   0.1190  33.7667 0.0000  3.7854  4.2519
A[T.30+]           -0.3589   0.1672  -2.1461 0.0319 -0.6867 -0.0311
S[T.5+]            -2.1474   0.0466 -46.0682 0.0000 -2.2387 -2.05

In [12]:
# create a contingency table for observed data
tab = df.pivot(index=['G', 'I', 'A'], columns='S', values='Count').sort_index()
print(tab ,'\n')

Itab = tab.groupby(level='I').sum().sum(axis=1)
print(Itab, '\n')

GItab = tab.groupby(level=['G', 'I']).sum().sum(axis=1).unstack('I')
print(GItab)

S                <5   5+
G     I   A             
<=260 no  <30    50    9
          30+    41    4
      yes <30   315   40
          30+   147   11
>260  no  <30    24    6
          30+    14    1
      yes <30  4012  459
          30+  1594  124 

I
no      149
yes    6702
dtype: int64 

I       no   yes
G               
<=260  104   513
>260    45  6189


In [14]:
# create a contingency table for fitted values
df['fitted_fm'] = fm.fittedvalues
tab_fm = df.pivot(index=['G', 'I', 'A'], columns='S', values='fitted_fm').sort_index()
print(tab_fm.round(2),'\n')

GItab_fm = tab_fm.groupby(level=['G', 'I']).sum().sum(axis=1).unstack('I')
print(GItab_fm.round(2))

# compute odds ratio
t = Table2x2(np.array(GItab))
t = Table2x2(np.array(GItab_fm))
lo, hi = t.oddsratio_confint()
print(t.oddsratio)      
print(lo, hi)   # 95% CI 

S                   <5      5+
G     I   A                   
<=260 no  <30    55.62    6.50
          30+    38.85    3.03
      yes <30   330.77   38.63
          30+   133.21   10.38
>260  no  <30    24.07    2.81
          30+    16.81    1.31
      yes <30  3990.54  466.06
          30+  1607.13  125.28 

I         no     yes
G                   
<=260  104.0   513.0
>260    45.0  6189.0
27.882001299544864
19.43072675599523 40.009105486904026


In [15]:
# collapse dataframe over A and S 
df1 = df.groupby(['G', 'I'], as_index=False)['Count'].sum()
print(df1, '\n')

# fit G * I to collapsed data
fm1 = glm(formula='Count ~ G * I', data=df1, family=Poisson()).fit()
print(fm1.params, '\n')
print(fm.params)

       G    I  Count
0  <=260   no    104
1  <=260  yes    513
2   >260   no     45
3   >260  yes   6189 

Intercept             4.644391
G[T.>260]            -0.837728
I[T.yes]              1.595885
G[T.>260]:I[T.yes]    3.327981
dtype: float64 

Intercept             4.018621
A[T.30+]             -0.358894
S[T.5+]              -2.147364
G[T.>260]            -0.837728
I[T.yes]              1.782807
A[T.30+]:S[T.5+]     -0.404311
A[T.30+]:I[T.yes]    -0.550584
G[T.>260]:I[T.yes]    3.327981
dtype: float64


c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\regression\_tools.py:133: RuntimeWarning: divide by zero encountered in scalar divide
  scale = np.dot(wresid, wresid) / df_resid
c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(


In [16]:
# fit G * I to original data
fm2 = glm(formula='Count ~ G * I', data=df, family=Poisson()).fit()
print(fm2.params)
df['fitted_fm2'] = fm2.fittedvalues
tab_fm2 = df.pivot(index=['G', 'I', 'A'], columns='S', values='fitted_fm2').sort_index()
print(tab_fm2,'\n')

Intercept             3.258097
G[T.>260]            -0.837728
I[T.yes]              1.595885
G[T.>260]:I[T.yes]    3.327981
dtype: float64
S                   <5       5+
G     I   A                    
<=260 no  <30    26.00    26.00
          30+    26.00    26.00
      yes <30   128.25   128.25
          30+   128.25   128.25
>260  no  <30    11.25    11.25
          30+    11.25    11.25
      yes <30  1547.25  1547.25
          30+  1547.25  1547.25 



In [17]:
# A and S
print(tab.groupby(level=['A']).sum())
print(tab_fm.groupby(level=['A']).sum())
print(tab_fm2.groupby(level=['A']).sum())

S      <5   5+
A             
<30  4401  514
30+  1796  140
S        <5     5+
A                 
<30  4401.0  514.0
30+  1796.0  140.0
S         <5       5+
A                    
<30  1712.75  1712.75
30+  1712.75  1712.75


In [18]:
# A
print(tab.groupby(level=['A']).sum().sum(axis=1))
print(tab_fm.groupby(level=['A']).sum().sum(axis=1))
print(tab_fm2.groupby(level=['A']).sum().sum(axis=1))

A
<30    4915
30+    1936
dtype: int64
A
<30    4915.0
30+    1936.0
dtype: float64
A
<30    3425.5
30+    3425.5
dtype: float64


In [19]:
# G and I
print(tab.groupby(level=['G', 'I']).sum().sum(axis=1).unstack('I'))
print(tab_fm.groupby(level=['G', 'I']).sum().sum(axis=1).unstack('I'))
print(tab_fm2.groupby(level=['G', 'I']).sum().sum(axis=1).unstack('I'))

I       no   yes
G               
<=260  104   513
>260    45  6189
I         no     yes
G                   
<=260  104.0   513.0
>260    45.0  6189.0
I         no     yes
G                   
<=260  104.0   513.0
>260    45.0  6189.0


In [20]:
# S
print(tab.sum())
print(tab_fm.sum())
print(tab_fm2.sum())

S
<5    6197
5+     654
dtype: int64
S
<5    6197.0
5+     654.0
dtype: float64
S
<5    3425.5
5+    3425.5
dtype: float64


In [21]:
# fm
tb_fm = np.array(tab_fm).reshape(2,2,2,2)
or_GI = (tb_fm[0,0,:,:] * tb_fm[1,1,:,:]) / (tb_fm[0,1,:,:] * tb_fm[1,0,:,:])
print(or_GI, '\n')

# fm2
tb_fm2 = np.array(tab_fm2).reshape(2,2,2,2)
or_GI = (tb_fm2[0,0,:,:] * tb_fm2[1,1,:,:]) / (tb_fm2[0,1,:,:] * tb_fm2[1,0,:,:])
print(or_GI, '\n')

[[27.8820013 27.8820013]
 [27.8820013 27.8820013]] 

[[27.8820013 27.8820013]
 [27.8820013 27.8820013]] 

